In [0]:
#################################################################
#              
# Autor: José Paulo
# Data: 2021-08-24
# Comentarios: Processo para carregar os dados para a bronze
##################################################################
#################################################################
#            Criando os Impports
#################################################################
from pyspark.sql.functions import current_timestamp,col, sum, when, trim
from functools import reduce

#################################################################
# Criando as variaveis dos paths para uso na aplicação   
#################################################################
resource_path     = '/Volumes/bikestore/resource/01-origem/'
bronze_path       = "bikestore.bronze"
silver_path       = "bikestore.silver"
silver_path_qua   = "bikestore.gold"
                
#################################################################
# Criando o mapeamento dos dados da camada Landing 
#################################################################                
bronze_map = {
        #"tmp_bronze_brands":      f"{resource_path}brands.csv",
        #"tmp_bronze_categories":  f"{resource_path}categories.csv/",
        "tmp_bronze_customers":   f"{resource_path}customers.csv/",
        #"tmp_bronze_order_items": f"{resource_path}order_items.csv/",
        #"tmp_bronze_orders":      f"{resource_path}orders.csv/",
        #"tmp_bronze_products":    f"{resource_path}products.csv/",
        #"tmp_bronze_staffs":      f"{resource_path}staffs.csv/",
        #"tmp_bronze_stocks":      f"{resource_path}stocks.csv/",
        #"tmp_bronze_stores":      f"{resource_path}stores.csv/",
}

validation_map = {
    "brands": ["brand_name"],
    "categories": ["category_name"],
    "customers": ["first_name", "last_name", "phone", "email"],
    #"customers": ["first_name", "last_name",  "email"],
    "order_items": ["item_id","product_id","quantity","list_price","discount"],
    "orders": ["customer_id","order_status","order_date","required_date","shipped_date","store_id","staff_id"],
    "products": ["product_name","brand_id","category_id","model_year","list_price"],
    "staffs": ["first_name","last_name","email","phone","active","store_id","manager_id"],
    "stocks": ["store_id","product_id","quantity"],
    "stores": ["store_id","store_name","phone","email","street","city","state","zip_code"],    
}
from pyspark.sql.functions import col, trim, lower
from functools import reduce

def validate_dataframe(df, table_name, validation_map):
    columns = validation_map.get(table_name, [])
    
    if not columns:
        raise ValueError(f"Nenhuma coluna de validação definida para {table_name}")

    dtypes = dict(df.dtypes)

    conditions = []

    for c in columns:
        dtype = dtypes[c]

        cond = col(c).isNull()

        if dtype == 'string':
            cond = cond | (trim(col(c)) == "") | (lower(col(c)) == "null")
        else:
            cond = cond | (col(c) == 0)

        conditions.append(cond)

    invalid_condition = reduce(lambda a, b: a | b, conditions)

    df_invalid = df.filter(invalid_condition)
    df_valid = df.filter(~invalid_condition)

    return df_valid, df_invalid
#################################################################
# Lendo cada arquivo da Landing Zone e gravando na camada bronze
#################################################################
for name, path in bronze_map.items():
    #################################################################
    # Lendo cada arquivo da Landing Zone e gerando o Dataframe
    #################################################################
    try:    
        df = ( 
            spark.read.csv(path, header=True, inferSchema=True)
                .withColumn("_ingestion_timestamp", current_timestamp())
        )

        
        nome_entidade = name.replace("tmp_bronze_", "")

        df_valido, df_invalido = validate_dataframe(df, nome_entidade , validation_map)

    except Exception as e:
        #######################################################
        # Falta colocaro processo de gravação de log de Erro
        #######################################################
        print(f"Error: {e}")
        dbutils.notebook.exit(f"Error: {e}")

    ########################################################################################
    # Gravando o DF de cada arquivoda landing Zone para dentro da Camada Bronze.
    ########################################################################################
    try:
        (
            df_valido.write
                    .format("delta")
                    .mode("overwrite")
                    .option("mergeSchema", "true") \
                    .saveAsTable(f"{silver_path}.{name.replace("tmp_bronze_", "")}")
        )
    except Exception as e:
        #######################################################
        # Falta colocaro processo de gravação de log de Erro
        #######################################################
        print(f"Error: {e}")
        dbutils.notebook.exit(f"Error: {e}")
    finally:
            ##################################################################################################
            # Falta colocaro processo de gravação de log de Processamento finalizado com sucesso    
            ##################################################################################################
            print(f"Sucesso ao gravar o arquivo {name} na camada Bronze")

    try:
        (
            df_invalido.write
                    .format("delta")
                    .mode("overwrite")
                    .option("mergeSchema", "true") \
                    .saveAsTable(f"{silver_path_qua}.{name.replace("tmp_bronze_", "")}_invalidos")
        )
    except Exception as e:
        #######################################################
        # Falta colocaro processo de gravação de log de Erro
        #######################################################
        print(f"Error: {e}")
        dbutils.notebook.exit(f"Error: {e}")
    finally:
            ##################################################################################################
            # Falta colocaro processo de gravação de log de Processamento finalizado com sucesso    
            ##################################################################################################
            print(f"Sucesso ao gravar o arquivo {name} na camada Bronze")
    